# 00 Run Everything - Publication Workflow

This is the single notebook entry point. Run this first for the complete Coswara pipeline and publication extras. The other notebooks are optional review/debug notebooks only.

Default behavior is conservative: expensive or optional stages are controlled by toggles, and existing outputs are skipped unless `FORCE_REBUILD = True`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Started: {datetime.now().isoformat(timespec='seconds')}")

## Configuration

Set only the paths and toggles you need. Keep `RUN_CNN = False` until the classical baseline pipeline is clean.

In [ ]:
RAW_COSWARA_DIR = PROJECT_ROOT / "data/raw/coswara"
COUGHVID_RAW = None  # Example: Path("/tmp/coughvid/public_dataset.zip") or Path("/data/coughvid")

FORCE_REBUILD = False
RUN_ENV_CHECK = True
RUN_LAYOUT_AUDIT = True

RUN_CORE_COSWARA = True
RUN_FEATURES = True
RUN_ML_BASELINES = True
RUN_CALIBRATION = True
RUN_FUSION = True
RUN_CNN = False
RUN_SHIFT_CHECKS = True
RUN_REPORT_ASSETS = True

RUN_PUBLICATION_EXTRAS = True
RUN_METADATA_BASELINE = True
RUN_QUALITY_WEIGHTED_FUSION = True
RUN_ABSTENTION = True
RUN_BOOTSTRAP_CI = True
RUN_COUGHVID_INDEX = COUGHVID_RAW is not None
RUN_COUGHVID_FEATURES = False
RUN_CROSS_DATASET = False
RUN_PAPER_TABLES = True
RUN_PAIRED_MODEL_COMPARISON = True
RUN_CONFOUNDING_MATCHING = True
RUN_FEATURE_SHIFT_REPORT = False
RUN_EXPERIMENT_MANIFEST = True

MIN_COUGH_DETECTED = 0.80
COUGHVID_FEATURE_MAX_ROWS = 25  # Use 25 for smoke test; set None for full extraction.
CNN_EPOCHS = 50
CNN_BATCH_SIZE = 32

print("Coswara:", RAW_COSWARA_DIR)
print("COUGHVID:", COUGHVID_RAW)

## Runner

The helper below runs project scripts, checks required inputs, and skips completed outputs unless forced.

In [ ]:
def p(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def path_exists(path):
    return p(path).exists() and p(path).stat().st_size > 0


def missing(paths):
    return [str(path) for path in paths if not path_exists(path)]


def run_step(name, args, enabled=True, requires=None, creates=None, strict_requires=True, force=None):
    requires = requires or []
    creates = creates or []
    force = FORCE_REBUILD if force is None else force
    if not enabled:
        print(f"SKIP {name}: disabled")
        return False
    missing_inputs = missing(requires)
    if missing_inputs:
        message = f"SKIP {name}: missing required input(s): {missing_inputs}"
        if strict_requires:
            raise FileNotFoundError(message)
        print(message)
        return False
    if creates and not force and all(path_exists(path) for path in creates):
        print(f"SKIP {name}: outputs already exist")
        return False
    cmd = [str(item) for item in args]
    print(f"
## {name}")
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return True


CORE_ARTIFACTS = [
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
    "data/processed/audio_quality.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/spectrogram_index.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]
print("Runner ready")

## Artifact Dashboard

This shows what already exists before the run.

In [ ]:
try:
    import pandas as pd
    dashboard = pd.DataFrame(
        [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
    )
    display(dashboard)
except Exception as exc:
    print(f"Dashboard unavailable before dependency install: {exc}")
    for path in CORE_ARTIFACTS:
        print(path, "OK" if path_exists(path) else "missing")

## Stage 0-1: Environment, Index, Splits, Quality

This stage is mandatory before modeling. It prevents training on bad audio, bad labels, or participant leakage.

In [ ]:
run_step("environment check", [sys.executable, "scripts/00_check_environment.py"], enabled=RUN_ENV_CHECK, force=True)

if RUN_CORE_COSWARA and not RAW_COSWARA_DIR.exists():
    raise FileNotFoundError(f"Coswara not found. Put it at {RAW_COSWARA_DIR} or change RAW_COSWARA_DIR.")

run_step(
    "raw layout audit",
    [sys.executable, "scripts/00_inspect_dataset_layout.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "reports/tables/coswara_layout_audit.csv"],
    enabled=RUN_CORE_COSWARA and RUN_LAYOUT_AUDIT,
    creates=["reports/tables/coswara_layout_audit.csv"],
)
run_step(
    "build Coswara index",
    [sys.executable, "scripts/01_build_coswara_index.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "data/interim/coswara_index.csv"],
    enabled=RUN_CORE_COSWARA,
    creates=["data/interim/coswara_index.csv"],
)
run_step(
    "clean metadata",
    [sys.executable, "scripts/02_clean_metadata.py", "--index", "data/interim/coswara_index.csv", "--output", "data/processed/metadata_clean.csv", "--availability-output", "data/interim/modality_availability.csv", "--audit-output", "reports/tables/dataset_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv"],
    creates=["data/processed/metadata_clean.csv", "data/interim/modality_availability.csv"],
)
run_step(
    "participant splits",
    [sys.executable, "scripts/03_create_splits.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/interim/split_manifest.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--audit-output", "reports/tables/split_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/interim/split_manifest.csv"],
)
run_step(
    "quality audit",
    [sys.executable, "scripts/04_quality_audit.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/audio_quality.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--summary-output", "reports/tables/quality_summary.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/split_manifest.csv", "data/processed/metadata_clean.csv"],
    creates=["data/processed/audio_quality.csv"],
)
run_step(
    "artifact validation",
    [sys.executable, "scripts/12_validate_artifacts.py", "--strict"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv", "data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    force=True,
)

## Hard Gate

If this cell fails, inspect the optional review notebooks before training models.

In [ ]:
import pandas as pd

from covid_audio_btp.notebook_utils import (
    assert_binary_labels_present,
    assert_no_participant_leakage,
    check_artifacts,
    read_optional_csv,
    stop_if_validation_errors,
)

metadata = pd.read_csv(p("data/processed/metadata_clean.csv"))
quality = pd.read_csv(p("data/processed/audio_quality.csv"))
issues = read_optional_csv("reports/tables/validation_issues.csv")

assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
if issues is not None:
    stop_if_validation_errors(issues)

quality_ok_rate = float((quality["quality_flag"] == "ok").mean()) if len(quality) else 0.0
print(f"quality ok rate: {quality_ok_rate:.3f}")
if quality_ok_rate < 0.50:
    raise AssertionError("Quality ok rate below 50%; review audio quality before modeling.")

## Stage 2-5: Features, Baselines, Calibration, Fusion

This is the main Coswara modeling path. CNN is optional and disabled by default.

In [ ]:
run_step(
    "feature and spectrogram extraction",
    [sys.executable, "scripts/05_extract_features.py", "--metadata", "data/processed/metadata_clean.csv", "--features-output", "data/processed/features_mfcc.csv", "--spectrogram-dir", "data/processed/spectrograms", "--spectrogram-index-output", "data/processed/spectrogram_index.csv", "--quality-mode", "all_samples"],
    enabled=RUN_FEATURES,
    requires=["data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    creates=["data/processed/features_mfcc.csv", "data/processed/spectrogram_index.csv"],
)
run_step(
    "classical ML baselines",
    [sys.executable, "scripts/06_train_ml_baselines.py", "--features", "data/processed/features_mfcc.csv"],
    enabled=RUN_ML_BASELINES,
    requires=["data/processed/features_mfcc.csv"],
    creates=["data/outputs/metrics/ml_baseline_metrics.csv", "data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
)
run_step(
    "compact CNN cough branch",
    [sys.executable, "scripts/07_train_cnn.py", "--spectrogram-index", "data/processed/spectrogram_index.csv", "--modality", "cough", "--epochs", CNN_EPOCHS, "--batch-size", CNN_BATCH_SIZE],
    enabled=RUN_CNN,
    requires=["data/processed/spectrogram_index.csv"],
    creates=["data/outputs/metrics/cnn_metrics.csv", "data/outputs/metrics/cnn_logits_validation.csv", "data/outputs/metrics/cnn_logits_test.csv"],
)
run_step(
    "branch calibration",
    [sys.executable, "scripts/08_calibrate_branches.py", "--validation-predictions", "data/outputs/metrics/ml_predictions_validation.csv", "--test-predictions", "data/outputs/metrics/ml_predictions_test.csv", "--method", "platt"],
    enabled=RUN_CALIBRATION,
    requires=["data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
    creates=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/calibration_metrics.csv"],
)
run_step(
    "standard fusion",
    [sys.executable, "scripts/09_run_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--validation-metrics", "data/outputs/metrics/ml_baseline_metrics.csv"],
    enabled=RUN_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["data/outputs/metrics/fusion_predictions.csv", "data/outputs/metrics/fusion_metrics.csv"],
)
run_step(
    "subgroup and confounding checks",
    [sys.executable, "scripts/10_shift_and_confounding_checks.py", "--predictions", "data/outputs/metrics/fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_SHIFT_CHECKS,
    requires=["data/outputs/metrics/fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/subgroup_metrics.csv"],
)
run_step(
    "basic report assets",
    [sys.executable, "scripts/11_make_report_assets.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/fusion_predictions.csv"],
    enabled=RUN_REPORT_ASSETS,
    requires=["data/processed/metadata_clean.csv"],
    creates=["reports/report_outline.md", "reports/slides_outline.md"],
)

## Stage 6: Publication Extras

These experiments support the publication-grade claims: metadata-only baseline, quality-weighted fusion, abstention, confidence intervals, COUGHVID external validation, and paper tables.

In [ ]:
run_step(
    "metadata-only baseline",
    [sys.executable, "scripts/14_train_metadata_baseline.py", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_METADATA_BASELINE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/metadata_baseline_metrics.csv", "data/outputs/metrics/metadata_predictions_validation.csv", "data/outputs/metrics/metadata_predictions_test.csv"],
)
run_step(
    "quality-weighted fusion",
    [sys.executable, "scripts/16_run_quality_weighted_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--quality", "data/processed/audio_quality.csv", "--validation-metrics", "data/outputs/metrics/ml_baseline_metrics.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_QUALITY_WEIGHTED_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/processed/audio_quality.csv", "data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/outputs/metrics/quality_weighted_fusion_metrics.csv"],
)
run_step(
    "abstention analysis",
    [sys.executable, "scripts/17_abstention_analysis.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_ABSTENTION,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/abstention_decisions.csv", "data/outputs/metrics/abstention_coverage_curve.csv"],
)
run_step(
    "bootstrap CI for quality-weighted fusion",
    [sys.executable, "scripts/15_bootstrap_prediction_metrics.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--output", "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv", "--group-columns", "fusion_method"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_BOOTSTRAP_CI,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv"],
)

if COUGHVID_RAW is not None:
    run_step(
        "COUGHVID index",
        [sys.executable, "scripts/13_build_coughvid_index.py", "--raw-dir", COUGHVID_RAW, "--output", "data/interim/coughvid_index.csv", "--min-cough-detected", MIN_COUGH_DETECTED],
        enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_INDEX,
        creates=["data/interim/coughvid_index.csv"],
    )
else:
    print("SKIP COUGHVID index: COUGHVID_RAW is None")

feature_cmd = [sys.executable, "scripts/19_extract_coughvid_features.py", "--index", "data/interim/coughvid_index.csv", "--features-output", "data/processed/coughvid_features_mfcc.csv", "--quality-ok-only"]
if COUGHVID_FEATURE_MAX_ROWS is not None:
    feature_cmd += ["--max-rows", COUGHVID_FEATURE_MAX_ROWS]
run_step(
    "COUGHVID feature extraction",
    feature_cmd,
    enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_FEATURES,
    requires=["data/interim/coughvid_index.csv"],
    creates=["data/processed/coughvid_features_mfcc.csv"],
    strict_requires=False,
)
run_step(
    "cross-dataset cough evaluation",
    [sys.executable, "scripts/18_cross_dataset_feature_eval.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv", "--modality", "cough", "--model-name", "logistic_regression"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CROSS_DATASET,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["data/outputs/metrics/cross_dataset_predictions.csv", "data/outputs/metrics/cross_dataset_metrics.csv"],
    strict_requires=False,
)
run_step(
    "paper metric tables",
    [sys.executable, "scripts/20_make_paper_tables.py"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAPER_TABLES,
    requires=["data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["reports/tables/paper_metric_table.csv", "reports/tables/paper_metric_table_raw.csv"],
    strict_requires=False,
)

## Stage 7: Extra Publication Strengthening

These optional diagnostics make the paper stronger: paired model comparison, matched confounding subset, source-vs-external feature shift, and a reproducibility manifest.

In [ ]:
run_step(
    "paired ML model comparison",
    [sys.executable, "scripts/21_paired_model_comparison.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--baseline-name", "logistic_regression", "--model-column", "model_name", "--group-columns", "modality", "--output", "data/outputs/metrics/paired_model_comparison.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAIRED_MODEL_COMPARISON,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv"],
    creates=["data/outputs/metrics/paired_model_comparison.csv"],
    strict_requires=False,
)
run_step(
    "confounding matched subset",
    [sys.executable, "scripts/22_confounding_matching.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--covariates", "age_bucket", "gender"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CONFOUNDING_MATCHING,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/metadata_matched.csv", "reports/tables/confounding_balance.csv"],
    strict_requires=False,
)
run_step(
    "feature shift report",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_FEATURE_SHIFT_REPORT,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["reports/tables/feature_shift_report.csv", "reports/tables/feature_shift_summary.csv"],
    strict_requires=False,
)
run_step(
    "experiment manifest",
    [sys.executable, "scripts/24_make_experiment_manifest.py", "--run-name", "covid_audio_publication_run"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_EXPERIMENT_MANIFEST,
    creates=["reports/experiment_manifest.json"],
)

## Final Dashboard

Use this at the end of the run to see exactly what was produced.

In [ ]:
import pandas as pd

final_dashboard = pd.DataFrame(
    [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
)
display(final_dashboard)

for path in [
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]:
    if path_exists(path):
        print(f"
{path}")
        display(pd.read_csv(p(path)).head(20))

print("Run finished")